# What Does TWFE Actually Do? Manual Demeaning and the FWL Theorem

**Author:** Carlos Mendez | **Website:** [carlos-mendez.org](https://carlos-mendez.org/)

---

## IMPORTANT: Verify the R Runtime

This notebook requires the **R runtime**, not Python. Google Colab defaults to Python, so you must switch to R before running any cells.

**How to activate the R runtime:**

1. Go to **Runtime** > **Change runtime type** in the menu bar
2. Under **Runtime type**, select **R**
3. Click **Save**
4. The runtime will restart with R

**How to verify:** Run the cell below. If it prints the R version, you are good to go. If you get a Python error, switch the runtime first.

---

In [ ]:
# Verify R runtime is active
cat("R version:", R.version.string, "\n")
cat("If you see a Python error, go to Runtime > Change runtime type > R\n")

## Install Additional Packages

Google Colab's R runtime comes with many packages pre-installed, including `tidyverse` and `scales`. The cell below installs only `fixest`, which is **not** pre-installed.

In [ ]:
# Only install packages NOT pre-installed in Google Colab R runtime
install.packages("fixest", repos = "https://cloud.r-project.org", quiet = TRUE)

In [ ]:
# Load libraries
library(fixest)
library(tidyverse)
library(scales)

set.seed(42)

# Site color palette
STEEL_BLUE   <- "#6a9bcc"
WARM_ORANGE  <- "#d97757"
NEAR_BLACK   <- "#141413"
TEAL         <- "#00d4c8"

# Variable labels for human-readable axis text
VAR_LABELS <- c(
  growth       = "GDP per Capita Growth",
  ln_y_initial = "Log Initial Income",
  log_s_k      = "Log Investment Share",
  log_n_gd     = "Log(n + g + d)",
  log_hcap     = "Log Human Capital",
  gov_cons     = "Gov. Consumption Share"
)

# Variables to demean (dependent + all regressors)
VARS_TO_DEMEAN <- c("growth", "ln_y_initial", "log_s_k",
                     "log_n_gd", "log_hcap", "gov_cons")

---

## 1. Load the Data from GitHub

We load a balanced panel dataset of 150 countries observed over 8 time periods from the Barro convergence exercise. The data is loaded directly from GitHub for full reproducibility --- no local files needed.

In [ ]:
# Load data directly from GitHub (reproducible in any environment)
DATA_URL <- "https://raw.githubusercontent.com/cmg777/starter-academic-v501/master/content/post/r_demeaning_twfe/referenceMaterials/barro_convergence_panel.csv"

panel_data <- read.csv(DATA_URL)
panel_data$id   <- factor(panel_data$id)
panel_data$time <- factor(panel_data$time)

n_countries <- nlevels(panel_data$id)
n_periods   <- nlevels(panel_data$time)

cat("Countries:", n_countries, "\n")
cat("Time periods:", n_periods, "\n")
cat("Total observations:", nrow(panel_data), "\n")
cat("Balanced panel:", all(table(panel_data$id) == n_periods), "\n\n")

cat("Summary of key variables:\n")
summary(panel_data[VARS_TO_DEMEAN])

---

## 2. TWFE Estimation with fixest

The `fixest` package absorbs both country and time fixed effects efficiently. The formula `| id + time` tells `feols()` to demean the data internally before estimating slope coefficients. Standard errors are clustered by entity (the default).

In [ ]:
twfe_model <- feols(
  growth ~ ln_y_initial + log_s_k + log_n_gd + log_hcap + gov_cons | id + time,
  data = panel_data
)

summary(twfe_model)

---

## 3. Manual Demeaning --- Step by Step

The two-way demeaning formula transforms each variable:

$$\tilde{x}_{it} = x_{it} - \bar{x}_{i \cdot} - \bar{x}_{\cdot t} + \bar{x}_{\cdot \cdot}$$

We subtract the **country mean** (persistent country differences), the **time mean** (common shocks), and add back the **grand mean** (to correct for the double subtraction).

In [ ]:
# Step 1: Country means (average over all periods for each country)
country_means <- panel_data |>
  group_by(id) |>
  summarise(across(all_of(VARS_TO_DEMEAN), mean), .groups = "drop")

# Step 2: Time means (average over all countries for each period)
time_means <- panel_data |>
  group_by(time) |>
  summarise(across(all_of(VARS_TO_DEMEAN), mean), .groups = "drop")

# Step 3: Grand mean (overall average)
grand_means <- colMeans(panel_data[VARS_TO_DEMEAN])
cat("Grand means:\n")
print(grand_means)

In [ ]:
# Step 4: Apply the demeaning formula programmatically
panel_dm <- panel_data |>
  left_join(
    country_means |> rename_with(~ paste0(.x, "_cmean"), all_of(VARS_TO_DEMEAN)),
    by = "id"
  ) |>
  left_join(
    time_means |> rename_with(~ paste0(.x, "_tmean"), all_of(VARS_TO_DEMEAN)),
    by = "time"
  )

for (v in VARS_TO_DEMEAN) {
  panel_dm[[paste0(v, "_dm")]] <-
    panel_dm[[v]] -
    panel_dm[[paste0(v, "_cmean")]] -
    panel_dm[[paste0(v, "_tmean")]] +
    grand_means[v]
}

# Verify demeaned means are approximately zero
cat("Mean of demeaned variables (should be ~0):\n")
dm_vars <- paste0(VARS_TO_DEMEAN, "_dm")
for (v in dm_vars) {
  cat(sprintf("  %-20s: %e\n", v, mean(panel_dm[[v]])))
}

---

## 4. OLS on the Demeaned Data

We now run plain OLS via `lm()` on the demeaned variables. No fixed effects machinery --- just ordinary least squares. If the FWL theorem holds, the slope coefficients should be **identical** to those from `feols()`.

In [ ]:
manual_model <- lm(
  growth_dm ~ ln_y_initial_dm + log_s_k_dm + log_n_gd_dm + log_hcap_dm + gov_cons_dm,
  data = panel_dm
)

summary(manual_model)

---

## 5. Coefficient Comparison: The Proof

The FWL theorem guarantees: $\hat{\beta}_{\text{TWFE}} = \hat{\beta}_{\text{OLS on demeaned data}}$

Let us verify this numerically.

In [ ]:
# Extract coefficients from both models
twfe_coefs   <- coef(twfe_model)
manual_coefs <- coef(manual_model)[-1]  # drop intercept
names(manual_coefs) <- names(twfe_coefs)

# Side-by-side comparison
comparison <- tibble(
  variable   = names(twfe_coefs),
  label      = VAR_LABELS[names(twfe_coefs)],
  feols_TWFE = twfe_coefs,
  manual_OLS = manual_coefs,
  difference = twfe_coefs - manual_coefs
)

cat("Side-by-side coefficient comparison:\n")
print(as.data.frame(comparison), digits = 12)

cat(sprintf("\nMaximum absolute difference: %e\n", max(abs(comparison$difference))))
cat("all.equal() test:", all.equal(unname(twfe_coefs), unname(manual_coefs)), "\n")

# Intercept check
intercept_val <- coef(manual_model)["(Intercept)"]
cat(sprintf("\nIntercept from demeaned OLS: %e (should be ~0)\n", intercept_val))

In [ ]:
# Coefficient comparison dot plot
comp_plot_data <- comparison |>
  pivot_longer(
    cols = c(feols_TWFE, manual_OLS),
    names_to = "method",
    values_to = "estimate"
  ) |>
  mutate(
    method = if_else(method == "feols_TWFE", "feols (TWFE)", "Manual Demeaning (OLS)"),
    label  = factor(label, levels = rev(VAR_LABELS[names(twfe_coefs)]))
  )

ggplot(comp_plot_data, aes(x = estimate, y = label, color = method, shape = method)) +
  geom_point(size = 4, position = position_dodge(width = 0.4)) +
  scale_color_manual(values = c("feols (TWFE)" = STEEL_BLUE,
                                "Manual Demeaning (OLS)" = WARM_ORANGE)) +
  scale_shape_manual(values = c("feols (TWFE)" = 16,
                                "Manual Demeaning (OLS)" = 17)) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray60") +
  labs(title = "Coefficient Comparison: TWFE vs Manual Demeaning",
       subtitle = "Coefficients are identical (FWL theorem)",
       x = "Coefficient Estimate", y = NULL, color = NULL, shape = NULL) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold", color = NEAR_BLACK),
        plot.subtitle = element_text(color = "gray40"),
        legend.position = "bottom")

---

## 6. Visualizing What Demeaning Does

Demeaning strips between-country differences and common time trends, leaving only **within-variation** --- the deviations that identify the TWFE coefficient.

In [ ]:
# Before vs after demeaning scatter (subset of 10 countries)
subset_ids <- levels(panel_dm$id)[1:10]
subset_data <- panel_dm |> filter(id %in% subset_ids)

raw_panel <- subset_data |>
  select(id, time, x = ln_y_initial, y = growth) |>
  mutate(panel = "Raw Data")

dm_panel <- subset_data |>
  select(id, time, x = ln_y_initial_dm, y = growth_dm) |>
  mutate(panel = "After Two-Way Demeaning")

scatter_long <- bind_rows(raw_panel, dm_panel) |>
  mutate(panel = factor(panel, levels = c("Raw Data", "After Two-Way Demeaning")))

ggplot(scatter_long, aes(x = x, y = y, color = id)) +
  geom_point(size = 2, alpha = 0.8) +
  facet_wrap(~ panel, scales = "free") +
  scale_color_viridis_d(option = "turbo", guide = "none") +
  labs(title = "What Demeaning Does: Raw vs Two-Way Demeaned Data",
       subtitle = "Demeaning strips between-country levels and common time trends",
       x = "Log Initial Income (raw / demeaned)",
       y = "GDP Growth (raw / demeaned)") +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold", color = NEAR_BLACK, size = 14),
        plot.subtitle = element_text(color = "gray40", size = 11),
        strip.text = element_text(face = "bold", size = 12))

In [ ]:
# Demeaning decomposition for Country 1
country_id <- "1"
country_sub <- panel_dm |> filter(id == country_id)

c_mean <- country_means |> filter(id == country_id) |> pull(growth)
t_means_vec <- time_means$growth
g_mean <- grand_means["growth"]

decomp_data <- tibble(
  period       = as.numeric(as.character(country_sub$time)),
  observed     = country_sub$growth,
  country_mean = c_mean,
  time_mean    = t_means_vec,
  grand_mean   = g_mean,
  demeaned     = country_sub$growth_dm
)

ggplot(decomp_data, aes(x = period)) +
  geom_line(aes(y = observed, color = "Observed (x_it)"), linewidth = 1.2) +
  geom_point(aes(y = observed, color = "Observed (x_it)"), size = 3) +
  geom_hline(aes(yintercept = country_mean, color = "Country mean (x_i.)"),
             linetype = "dashed", linewidth = 0.9) +
  geom_line(aes(y = time_mean, color = "Time mean (x_.t)"),
            linetype = "dotdash", linewidth = 0.9) +
  geom_point(aes(y = time_mean, color = "Time mean (x_.t)"), size = 2) +
  geom_hline(aes(yintercept = grand_mean, color = "Grand mean (x_..)"),
             linetype = "dotted", linewidth = 0.9) +
  geom_line(aes(y = demeaned, color = "Demeaned (x_tilde_it)"), linewidth = 1.2) +
  geom_point(aes(y = demeaned, color = "Demeaned (x_tilde_it)"), size = 3) +
  scale_color_manual(values = c(
    "Observed (x_it)"       = STEEL_BLUE,
    "Country mean (x_i.)"   = WARM_ORANGE,
    "Time mean (x_.t)"      = TEAL,
    "Grand mean (x_..)"     = "gray50",
    "Demeaned (x_tilde_it)" = NEAR_BLACK
  )) +
  scale_x_continuous(breaks = 1:8) +
  labs(title = paste("Demeaning Decomposition: Country", country_id, "(Growth)"),
       x = "Time Period", y = "Growth Rate", color = NULL) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold", color = NEAR_BLACK),
        legend.position = "bottom") +
  guides(color = guide_legend(nrow = 2))

---

## 7. A Caveat: Standard Errors Differ

While the coefficients are identical, the **standard errors** from `lm()` are wrong. Naive `lm()` ignores the 157 degrees of freedom consumed by estimating 150 country effects and 8 time effects. The correct df is 1,038, not 1,195.

In [ ]:
# Three types of standard errors
manual_se_naive <- summary(manual_model)$coefficients[-1, "Std. Error"]
names(manual_se_naive) <- names(twfe_coefs)

se_comparison <- tibble(
  variable         = names(twfe_coefs),
  label            = VAR_LABELS[names(twfe_coefs)],
  se_naive_lm      = manual_se_naive,
  se_feols_iid     = se(twfe_model, se = "iid"),
  se_feols_cluster = se(twfe_model)
)

cat("Standard error comparison:\n")
print(as.data.frame(se_comparison), digits = 6)

cat("\nWhy SEs differ:\n")
cat("- Naive lm() uses df = N*T - K =", nrow(panel_data) - length(manual_coefs), "\n")
cat("- Correct df = N*T - N - T + 1 - K =",
    nrow(panel_data) - n_countries - n_periods + 1 - length(twfe_coefs), "\n")

In [ ]:
# SE comparison bar chart
se_plot_data <- se_comparison |>
  pivot_longer(cols = starts_with("se_"), names_to = "se_type", values_to = "se_value") |>
  mutate(
    se_type = case_match(se_type,
      "se_naive_lm"      ~ "Naive lm()",
      "se_feols_iid"     ~ "feols (IID)",
      "se_feols_cluster" ~ "feols (Clustered)"
    ),
    se_type = factor(se_type, levels = c("Naive lm()", "feols (IID)", "feols (Clustered)")),
    label   = factor(label, levels = VAR_LABELS[names(twfe_coefs)])
  )

ggplot(se_plot_data, aes(x = label, y = se_value, fill = se_type)) +
  geom_col(position = position_dodge(width = 0.7), width = 0.6) +
  scale_fill_manual(values = c("Naive lm()" = "gray70",
                                "feols (IID)" = WARM_ORANGE,
                                "feols (Clustered)" = STEEL_BLUE)) +
  labs(title = "Standard Error Comparison: Why Naive SEs Are Wrong",
       subtitle = "Naive lm() ignores absorbed degrees of freedom",
       x = NULL, y = "Standard Error", fill = NULL) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold", color = NEAR_BLACK),
        plot.subtitle = element_text(color = "gray40", size = 10),
        axis.text.x = element_text(angle = 25, hjust = 1),
        legend.position = "bottom")

---

## Conclusion

This notebook demonstrated that **TWFE is just OLS on demeaned data**. The FWL theorem guarantees exact coefficient equivalence (max difference: ~3 x 10^-16). However, naive standard errors from `lm()` are wrong because they ignore the degrees of freedom consumed by the fixed effects --- always use a dedicated panel estimator like `fixest` for inference.

**Key takeaways:**
1. Demeaning formula: subtract country means, subtract time means, add back the grand mean
2. Coefficients match to machine precision (`all.equal() = TRUE`)
3. Naive SEs understate uncertainty by 7--22%
4. Within R-squared (0.177) vs overall (0.755) shows fixed effects absorb most variation

---

*Full tutorial with detailed explanations: [carlos-mendez.org/post/r_demeaning_twfe/](https://carlos-mendez.org/post/r_demeaning_twfe/)*